In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohdtahasayed2904/weqetdf/Malware_Dataset.csv


In [2]:
# ================================================================
# COMPLETE DATASET AUDIT FOR DEEPSEEK-CODER-1.3B-BASE
# Automatically discovers and inspects CSV files in Kaggle Input
# ================================================================

import os
import glob
import hashlib
import warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 250)

print("=" * 100)
print("                 DEEPSEEK DATASET — COMPLETE AUDIT")
print("=" * 100)

# ------------------------------------------------
# 1. FIND ALL CSV FILES
# ------------------------------------------------

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

if not csv_files:
    raise FileNotFoundError("No CSV files found under /kaggle/input/")

print(f"\nTOTAL CSV FILES FOUND: {len(csv_files)}")

for i, path in enumerate(csv_files, 1):
    size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"\n{i}. {path}")
    print(f"   Size: {size_mb:.2f} MB")

# ------------------------------------------------
# 2. INSPECT EVERY CSV
# ------------------------------------------------

datasets = {}

for file_index, path in enumerate(csv_files, 1):

    filename = os.path.basename(path)

    print("\n\n" + "=" * 100)
    print(f"DATASET {file_index}: {filename}")
    print("=" * 100)

    try:
        df = pd.read_csv(path, low_memory=False)
        datasets[path] = df

        rows, cols = df.shape

        print("\n[ A. BASIC INFORMATION ]")
        print("File path:", path)
        print("Rows:", f"{rows:,}")
        print("Columns:", f"{cols:,}")
        print("Memory usage:", f"{df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
        print("Duplicate rows:", f"{df.duplicated().sum():,}")

        print("\n[ B. ALL COLUMN NAMES ]")
        for i, col in enumerate(df.columns, 1):
            print(f"{i:5}. {col}")

        print("\n[ C. DATA TYPES ]")
        print(df.dtypes.value_counts().to_string())
        print("\nDetailed data types:")
        print(df.dtypes.to_string())

        print("\n[ D. MISSING VALUES ]")
        missing = df.isnull().sum()
        missing = missing[missing > 0].sort_values(ascending=False)

        if len(missing):
            print(missing.to_string())
            print("\nTotal missing cells:", f"{df.isnull().sum().sum():,}")
        else:
            print("No missing values.")

        print("\n[ E. UNIQUE VALUES PER COLUMN ]")
        for col in df.columns:
            try:
                print(f"{col}: {df[col].nunique(dropna=True):,} unique values")
            except Exception:
                pass

        print("\n[ F. POSSIBLE IDENTIFIER / LABEL COLUMNS ]")

        for col in df.columns:
            name = str(col).lower()

            if any(x in name for x in [
                "sha", "hash", "md5", "id", "uuid",
                "label", "type", "class", "family",
                "category", "target", "malware"
            ]):
                print(
                    f"{col} | dtype={df[col].dtype} | "
                    f"unique={df[col].nunique(dropna=True):,}"
                )

                if df[col].nunique(dropna=True) <= 50:
                    print("  Value distribution:")
                    print(df[col].value_counts(dropna=False).head(50).to_string())

        print("\n[ G. POSSIBLE CLASS/LABEL DISTRIBUTIONS ]")

        label_candidates = [
            col for col in df.columns
            if any(x in str(col).lower() for x in [
                "label", "type", "class", "family",
                "category", "target", "malware"
            ])
        ]

        for col in label_candidates:
            if df[col].nunique(dropna=True) <= 100:
                print(f"\nColumn: {col}")
                print(df[col].value_counts(dropna=False).to_string())

        print("\n[ H. NUMERICAL FEATURE SUMMARY ]")

        numeric_df = df.select_dtypes(include=[np.number])

        if not numeric_df.empty:
            print("Number of numeric columns:", len(numeric_df.columns))
            print("\nDescriptive statistics:")
            print(numeric_df.describe().T.to_string())

            print("\nConstant numerical columns:")
            constant_cols = [
                col for col in numeric_df.columns
                if numeric_df[col].nunique(dropna=False) <= 1
            ]
            print(constant_cols if constant_cols else "None")

            print("\nNumerical columns with zero values:")
            zero_cols = [
                col for col in numeric_df.columns
                if (numeric_df[col] == 0).any()
            ]
            print(f"{len(zero_cols)} columns contain at least one zero.")

        else:
            print("No numerical columns.")

        print("\n[ I. CATEGORICAL FEATURE SUMMARY ]")

        categorical_df = df.select_dtypes(
            include=["object", "category", "bool"]
        )

        print("Number of categorical columns:", len(categorical_df.columns))

        for col in categorical_df.columns[:100]:
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).head(15).to_string())

        print("\n[ J. FIRST 5 ROWS ]")
        print(df.head(5).to_string())

        print("\n[ K. RANDOM 5 ROWS ]")
        print(df.sample(min(5, len(df)), random_state=42).to_string())

        print("\n[ L. LAST 5 ROWS ]")
        print(df.tail(5).to_string())

        print("\n[ M. FEATURE NAME GROUPS — AUTOMATIC KEYWORD SCAN ]")

        groups = {
            "API-related": ["api", "function"],
            "DLL-related": ["dll", "library", "import"],
            "PE Header": ["header", "machine", "optional", "characteristic"],
            "PE Sections": ["section", "virtual", "raw", "entropy"],
            "Identifiers": ["sha", "hash", "md5", "uuid"],
            "Labels": ["label", "type", "class", "family", "target", "malware"]
        }

        for group, keywords in groups.items():
            matched = [
                col for col in df.columns
                if any(keyword in str(col).lower() for keyword in keywords)
            ]

            print(f"\n{group}: {len(matched)} matching columns")
            if matched:
                print(matched[:200])

        print("\n[ N. DATASET INTEGRITY CHECKS ]")

        print("Rows with all values missing:", f"{df.isnull().all(axis=1).sum():,}")
        print("Columns with all values missing:", f"{df.isnull().all(axis=0).sum():,}")

        if len(df.columns):
            print("Potential duplicate identifier columns:")

            identifier_cols = [
                col for col in df.columns
                if any(x in str(col).lower() for x in ["sha256", "sha", "hash", "md5"])
            ]

            if identifier_cols:
                for col in identifier_cols:
                    print(
                        f"{col}: {df[col].nunique(dropna=True):,} unique / "
                        f"{df[col].isna().sum():,} missing / "
                        f"{df[col].duplicated().sum():,} duplicated"
                    )
            else:
                print("No obvious hash column detected.")

    except Exception as e:
        print(f"\nERROR READING {path}: {repr(e)}")

# ------------------------------------------------
# 3. FINAL SUMMARY
# ------------------------------------------------

print("\n\n" + "=" * 100)
print("                         FINAL AUDIT SUMMARY")
print("=" * 100)

print("CSV files successfully loaded:", len(datasets))

for path, df in datasets.items():
    print("\nFile:", os.path.basename(path))
    print("Shape:", df.shape)
    print("Columns:", len(df.columns))
    print("Rows:", len(df))
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing cells:", df.isnull().sum().sum())

print("\n" + "=" * 100)
print("AUDIT COMPLETE")
print("=" * 100)

print("""
IMPORTANT:
1. No data has been modified.
2. No rows or columns have been removed.
3. No train/validation/test split has been performed.
4. All successfully loaded DataFrames are available in `datasets`.
5. Share the complete output so we can design the DeepSeek preprocessing
   and masked-SFT pipeline based on the actual source data.
""")

                 DEEPSEEK DATASET — COMPLETE AUDIT

TOTAL CSV FILES FOUND: 1

1. /kaggle/input/datasets/mohdtahasayed2904/weqetdf/Malware_Dataset.csv
   Size: 24.19 MB


DATASET 1: Malware_Dataset.csv

[ A. BASIC INFORMATION ]
File path: /kaggle/input/datasets/mohdtahasayed2904/weqetdf/Malware_Dataset.csv
Rows: 29,489
Columns: 320
Memory usage: 74.95 MB
Duplicate rows: 0

[ B. ALL COLUMN NAMES ]
    1. SHA256
    2. Type
    3. virtualallocex
    4. setunhandledexceptionfilter
    5. getprocaddress
    6. getcurrentprocessid
    7. setlasterror
    8. multibytetowidechar
    9. createfilew
   10. getmodulefilenamew
   11. sleep
   12. widechartomultibyte
   13. gettickcount
   14. getmodulehandlew
   15. closehandle
   16. getmodulehandlea
   17. raiseexception
   18. sizeofresource
   19. loadresource
   20. getversionexa
   21. getlasterror
   22. getcommandlinew
   23. heapfree
   24. isdebuggerpresent
   25. getcurrentthreadid
   26. exitprocess
   27. getstdhandle
   28. writefile

In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# ============================================================
# 1. LOAD DATASET
# ============================================================

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/weqetdf/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 65)
print("DATASET LOADED")
print("=" * 65)

print("Shape:", df.shape)
print("Memory usage:", round(df.memory_usage(deep=True).sum() / 1024**2, 2), "MB")


# ============================================================
# 2. VALIDATE REQUIRED COLUMNS
# ============================================================

assert "SHA256" in df.columns, "SHA256 column missing!"
assert "Type" in df.columns, "Type target column missing!"

assert df["SHA256"].notna().all(), "Missing SHA256 values found!"
assert df["Type"].notna().all(), "Missing target labels found!"

assert df["SHA256"].is_unique, "Duplicate SHA256 hashes found!"

feature_cols = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

assert len(feature_cols) == 318, (
    f"Expected 318 features, found {len(feature_cols)}"
)

assert df[feature_cols].isnull().sum().sum() == 0, (
    "Missing feature values found!"
)

non_numeric = df[feature_cols].select_dtypes(
    exclude=[np.number]
).columns.tolist()

assert len(non_numeric) == 0, (
    f"Non-numeric features found: {non_numeric}"
)

print("\nDataset validation: PASSED")
print("Feature count:", len(feature_cols))
print("Unique SHA256 hashes:", df["SHA256"].nunique())


# ============================================================
# 3. VERIFY FEATURE GROUPS
# ============================================================

# Expected feature ordering from our previous dataset analysis.
api_cols = feature_cols[:200]
dll_cols = feature_cols[200:227]
pe_header_cols = feature_cols[227:273]
pe_section_cols = feature_cols[273:318]

feature_groups = {
    "API Functions": api_cols,
    "DLL Imports": dll_cols,
    "PE Header": pe_header_cols,
    "PE Sections": pe_section_cols
}

print("\n" + "=" * 65)
print("FEATURE GROUP VERIFICATION")
print("=" * 65)

for group_name, columns in feature_groups.items():
    print(f"\n{group_name}: {len(columns)} features")
    print("First 5:", columns[:5])
    print("Last 5:", columns[-5:])

assert sum(len(v) for v in feature_groups.values()) == 318


# ============================================================
# 4. LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 65)
print("TARGET LABEL DISTRIBUTION")
print("=" * 65)

label_distribution = (
    df["Type"]
    .value_counts()
    .sort_index()
    .rename_axis("Type")
    .to_frame("Samples")
)

label_distribution["Percentage"] = (
    label_distribution["Samples"] / len(df) * 100
).round(2)

display(label_distribution)

print("Unique labels:", sorted(df["Type"].unique()))


# ============================================================
# 5. REPRODUCIBLE STRATIFIED 60:20:20 SPLIT
# ============================================================

# First: 80% development, 20% test
development_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Type"]
)

# Second: Split development into 60% train and 20% validation
train_df, validation_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=42,
    stratify=development_df["Type"]
)

# Reset indices
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


# ============================================================
# 6. VERIFY SPLIT
# ============================================================

print("\n" + "=" * 65)
print("FINAL DATASET SPLIT")
print("=" * 65)

print(f"Training samples:   {len(train_df):,}")
print(f"Validation samples: {len(validation_df):,}")
print(f"Test samples:       {len(test_df):,}")
print(f"Total samples:      {len(train_df) + len(validation_df) + len(test_df):,}")

assert len(train_df) == 17693
assert len(validation_df) == 5898
assert len(test_df) == 5898

# Check that no SHA256 appears in multiple splits
train_hashes = set(train_df["SHA256"])
val_hashes = set(validation_df["SHA256"])
test_hashes = set(test_df["SHA256"])

assert train_hashes.isdisjoint(val_hashes)
assert train_hashes.isdisjoint(test_hashes)
assert val_hashes.isdisjoint(test_hashes)

print("\nNo SHA256 overlap between splits: PASSED")


# ============================================================
# 7. VERIFY STRATIFICATION
# ============================================================

print("\nClass distribution across splits:")

for split_name, split_df in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{split_name}:")
    print(
        split_df["Type"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )


# ============================================================
# 8. SAVE SPLITS
# ============================================================

OUTPUT_DIR = "/kaggle/working/deepseek_dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_df.to_csv(
    f"{OUTPUT_DIR}/train.csv",
    index=False
)

validation_df.to_csv(
    f"{OUTPUT_DIR}/validation.csv",
    index=False
)

test_df.to_csv(
    f"{OUTPUT_DIR}/test.csv",
    index=False
)

print("\n" + "=" * 65)
print("STEP 1 COMPLETED SUCCESSFULLY")
print("=" * 65)

print("Saved files:")
print(f"{OUTPUT_DIR}/train.csv")
print(f"{OUTPUT_DIR}/validation.csv")
print(f"{OUTPUT_DIR}/test.csv")

DATASET LOADED
Shape: (29489, 320)
Memory usage: 74.95 MB

Dataset validation: PASSED
Feature count: 318
Unique SHA256 hashes: 29489

FEATURE GROUP VERIFICATION

API Functions: 200 features
First 5: ['virtualallocex', 'setunhandledexceptionfilter', 'getprocaddress', 'getcurrentprocessid', 'setlasterror']
Last 5: ['vbavartstne', 'vbalatememcall', 'buildcommdcbandtimeoutsa', 'getconsolealiaseslengtha', 'vbastrvarval']

DLL Imports: 27 features
First 5: ['advapi32.dll', 'kernel32.dll', 'ole32.dll', 'oleaut32.dll', 'shlwapi.dll']
Last 5: ['oleacc.dll', 'msvcp90.dll', 'msvbvm60.dll', 'mfc42.dll', 'shfolder.dll']

PE Header: 46 features
First 5: ['e_cblp', 'e_cp', 'e_crlc', 'e_cparhdr', 'e_minalloc']
Last 5: ['SizeOfHeaders', 'CheckSum', 'Subsystem', 'DllCharacteristics', 'SizeOfStackReserve']

PE Sections: 45 features
First 5: ['text_Misc_VirtualSize', 'text_VirtualAddress', 'text_SizeOfRawData', 'text_PointerToRawData', 'text_Characteristics']
Last 5: ['pdata_Misc_VirtualSize', 'pdata_Virt

,Samples,Percentage
Type,,
0,1877,6.37
1,5022,17.03
2,4643,15.74
3,4955,16.80
4,5076,17.21
5,4217,14.30
6,3699,12.54


Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

FINAL DATASET SPLIT
Training samples:   17,693
Validation samples: 5,898
Test samples:       5,898
Total samples:      29,489

No SHA256 overlap between splits: PASSED

Class distribution across splits:

Train:
Type
0     6.37
1    17.03
2    15.74
3    16.80
4    17.22
5    14.30
6    12.54
Name: proportion, dtype: float64

Validation:
Type
0     6.36
1    17.02
2    15.75
3    16.80
4    17.21
5    14.31
6    12.55
Name: proportion, dtype: float64

Test:
Type
0     6.36
1    17.04
2    15.75
3    16.80
4    17.21
5    14.29
6    12.55
Name: proportion, dtype: float64

STEP 1 COMPLETED SUCCESSFULLY
Saved files:
/kaggle/working/deepseek_dataset/train.csv
/kaggle/working/deepseek_dataset/validation.csv
/kaggle/working/deepseek_dataset/test.csv


In [4]:
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed2904/weqetdf/Malware_Dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Available target labels:")
print(sorted(df["Type"].unique()))

print("\nLabel counts:")
print(df["Type"].value_counts().sort_index())

print("\nDataset metadata:")
print("Columns:", df.columns.tolist()[:10])
print("Target column:", "Type")

Available target labels:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Label counts:
Type
0    1877
1    5022
2    4643
3    4955
4    5076
5    4217
6    3699
Name: count, dtype: int64

Dataset metadata:
Columns: ['SHA256', 'Type', 'virtualallocex', 'setunhandledexceptionfilter', 'getprocaddress', 'getcurrentprocessid', 'setlasterror', 'multibytetowidechar', 'createfilew', 'getmodulefilenamew']
Target column: Type


In [5]:
import os
import json
import numpy as np
import pandas as pd

from transformers import AutoTokenizer
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

DATA_DIR = "/kaggle/working/deepseek_dataset"

TRAIN_PATH = f"{DATA_DIR}/train.csv"
VAL_PATH = f"{DATA_DIR}/validation.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

OUTPUT_DIR = "/kaggle/working/deepseek_prompts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Confirmed label mapping from our previous experiment
LABEL_MAP = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

# ============================================================
# 2. LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("=" * 65)
print("DATA LOADED")
print("=" * 65)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# ============================================================
# 3. DEFINE FEATURE GROUPS
# ============================================================

feature_cols = [
    col for col in train_df.columns
    if col not in ["SHA256", "Type"]
]

api_cols = feature_cols[:200]
dll_cols = feature_cols[200:227]
pe_header_cols = feature_cols[227:273]
pe_section_cols = feature_cols[273:318]

FEATURE_GROUPS = {
    "API_FUNCTIONS": api_cols,
    "DLL_IMPORTS": dll_cols,
    "PE_HEADER": pe_header_cols,
    "PE_SECTIONS": pe_section_cols
}

assert len(feature_cols) == 318
assert len(api_cols) == 200
assert len(dll_cols) == 27
assert len(pe_header_cols) == 46
assert len(pe_section_cols) == 45

# ============================================================
# 4. LOAD DEEPSEEK TOKENIZER
# ============================================================

print("\nLoading DeepSeek tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

print("Tokenizer loaded successfully.")

# ============================================================
# 5. FEATURE VALUE FORMATTER
# ============================================================

def format_value(value):
    """
    Preserve the actual numerical value.
    No conversion of zero values into missing/present indicators.
    """

    if pd.isna(value):
        raise ValueError("Missing feature value encountered.")

    if isinstance(value, (np.integer, int)):
        return str(int(value))

    if isinstance(value, (np.floating, float)):
        if float(value).is_integer():
            return str(int(value))
        return repr(float(value))

    return str(value)


# ============================================================
# 6. BUILD COMPLETE FEATURE PROFILE
# ============================================================

def build_feature_profile(row):

    sections = []

    for group_name, columns in FEATURE_GROUPS.items():

        lines = [
            f"{column}={format_value(row[column])}"
            for column in columns
        ]

        section = (
            f"<{group_name}>\n"
            + "\n".join(lines)
            + f"\n</{group_name}>"
        )

        sections.append(section)

    return "\n\n".join(sections)


# ============================================================
# 7. BUILD DEEPSEEK PROMPT
# ============================================================

SYSTEM_INSTRUCTION = """You are a Windows PE malware classification model.

Your task is to classify a Windows executable using its provided static PE features.

Rules:
1. Analyze only the supplied static feature values.
2. Consider all API function values, DLL import values, PE header values, and PE section values.
3. Zero values are meaningful and must not be ignored.
4. Do not assume runtime behavior that is not supported by the features.
5. Select exactly one malware family from the permitted classes.
6. Return the classification in the required output format.
"""

def build_prompt(row):

    feature_profile = build_feature_profile(row)

    prompt = f"""### SYSTEM:
{SYSTEM_INSTRUCTION}

### USER:
Analyze the following Windows PE static feature profile.

{feature_profile}

Classify this executable into exactly one of these classes:
Benign, RedLineStealer, Downloader, RAT, BankingTrojan, SnakeKeyLogger, Spyware.

Return your answer using this exact format:

Predicted Malware Family: <class name>

### ASSISTANT:
"""

    return prompt


def build_completion(label_id):

    label_id = int(label_id)

    if label_id not in LABEL_MAP:
        raise ValueError(f"Unknown label ID: {label_id}")

    return f"Predicted Malware Family: {LABEL_MAP[label_id]}"


# ============================================================
# 8. CREATE PROMPT DATASETS
# ============================================================

def create_examples(dataframe, split_name):

    examples = []

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=f"Building {split_name}"
    ):

        prompt = build_prompt(row)
        completion = build_completion(row["Type"])

        examples.append({
            "prompt": prompt,
            "completion": completion,
            "label_id": int(row["Type"]),
            "label_name": LABEL_MAP[int(row["Type"])]
        })

    return examples


train_examples = create_examples(train_df, "Training")
val_examples = create_examples(val_df, "Validation")
test_examples = create_examples(test_df, "Testing")

# ============================================================
# 9. SAVE PROMPT DATASETS
# ============================================================

def save_jsonl(examples, filename):

    path = os.path.join(OUTPUT_DIR, filename)

    with open(path, "w", encoding="utf-8") as file:

        for example in examples:
            file.write(json.dumps(example, ensure_ascii=False) + "\n")

    print(f"Saved: {path}")


save_jsonl(train_examples, "train_prompts.jsonl")
save_jsonl(val_examples, "validation_prompts.jsonl")
save_jsonl(test_examples, "test_prompts.jsonl")

# ============================================================
# 10. TOKEN-LENGTH ANALYSIS
# ============================================================

print("\nAnalyzing token lengths...")

def get_token_lengths(examples):

    lengths = []

    for example in tqdm(examples):

        full_text = (
            example["prompt"]
            + example["completion"]
            + tokenizer.eos_token
        )

        token_ids = tokenizer(
            full_text,
            add_special_tokens=False
        )["input_ids"]

        lengths.append(len(token_ids))

    return np.array(lengths)


train_lengths = get_token_lengths(train_examples)

print("\n" + "=" * 65)
print("TRAINING TOKEN-LENGTH ANALYSIS")
print("=" * 65)

stats = {
    "Minimum": np.min(train_lengths),
    "Median": np.median(train_lengths),
    "Mean": np.mean(train_lengths),
    "90th percentile": np.percentile(train_lengths, 90),
    "95th percentile": np.percentile(train_lengths, 95),
    "99th percentile": np.percentile(train_lengths, 99),
    "Maximum": np.max(train_lengths)
}

for name, value in stats.items():
    print(f"{name:20s}: {value:,.0f}")

# ============================================================
# 11. FEATURE PRESERVATION CHECK
# ============================================================

sample_row = train_df.iloc[0]

sample_profile = build_feature_profile(sample_row)

preserved_feature_count = sum(
    1 for column in feature_cols
    if f"{column}=" in sample_profile
)

zero_feature_count = sum(
    1 for column in feature_cols
    if float(sample_row[column]) == 0
)

print("\n" + "=" * 65)
print("FEATURE PRESERVATION CHECK")
print("=" * 65)

print("Expected features:", len(feature_cols))
print("Features represented:", preserved_feature_count)
print("Zero-valued features:", zero_feature_count)

assert preserved_feature_count == 318

# Check SHA256 does not appear in the prompt
assert "SHA256" not in build_prompt(sample_row)

# Check every zero-valued feature is explicitly represented
for column in feature_cols:
    if float(sample_row[column]) == 0:
        assert f"{column}=0" in sample_profile

print("All feature preservation checks passed.")

# ============================================================
# 12. DISPLAY EXAMPLE
# ============================================================

print("\n" + "=" * 65)
print("EXAMPLE PROMPT PREVIEW")
print("=" * 65)

print(train_examples[0]["prompt"][:4000])

print("\nExpected completion:")
print(train_examples[0]["completion"])

# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("STEP 2 COMPLETED")
print("=" * 65)

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Testing examples:", len(test_examples))

print("Feature values preserved:", 318)
print("SHA256 excluded: Yes")
print("Zero values retained: Yes")
print("Class mapping applied: Yes")

print("\nPrompt files saved in:", OUTPUT_DIR)

DATA LOADED
Train: (17693, 320)
Validation: (5898, 320)
Test: (5898, 320)

Loading DeepSeek tokenizer...


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

Tokenizer loaded successfully.


... (2 outputs left out)

Building Testing:   0%|          | 0/5898 [00:00<?, ?it/s]

Saved: /kaggle/working/deepseek_prompts/train_prompts.jsonl
Saved: /kaggle/working/deepseek_prompts/validation_prompts.jsonl
Saved: /kaggle/working/deepseek_prompts/test_prompts.jsonl

Analyzing token lengths...


  0%|          | 0/17693 [00:00<?, ?it/s]


TRAINING TOKEN-LENGTH ANALYSIS
Minimum             : 2,770
Median              : 2,851
Mean                : 2,858
90th percentile     : 2,895
95th percentile     : 2,918
99th percentile     : 2,977
Maximum             : 3,016

FEATURE PRESERVATION CHECK
Expected features: 318
Features represented: 318
Zero-valued features: 171
All feature preservation checks passed.

EXAMPLE PROMPT PREVIEW
### SYSTEM:
You are a Windows PE malware classification model.

Your task is to classify a Windows executable using its provided static PE features.

Rules:
1. Analyze only the supplied static feature values.
2. Consider all API function values, DLL import values, PE header values, and PE section values.
3. Zero values are meaningful and must not be ignored.
4. Do not assume runtime behavior that is not supported by the features.
5. Select exactly one malware family from the permitted classes.
6. Return the classification in the required output format.


### USER:
Analyze the following Windows PE s

In [6]:
import os
import json
import torch
import numpy as np

from datasets import Dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

PROMPT_DIR = "/kaggle/working/deepseek_prompts"
OUTPUT_DIR = "/kaggle/working/deepseek_tokenized"

os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LENGTH = 4096

TRAIN_FILE = f"{PROMPT_DIR}/train_prompts.jsonl"
VAL_FILE = f"{PROMPT_DIR}/validation_prompts.jsonl"
TEST_FILE = f"{PROMPT_DIR}/test_prompts.jsonl"

# ============================================================
# 2. LOAD TOKENIZER
# ============================================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.eos_token is None:
    raise ValueError("Tokenizer has no EOS token.")

EOS_TOKEN = tokenizer.eos_token
EOS_TOKEN_ID = tokenizer.eos_token_id

print("EOS token:", repr(EOS_TOKEN))
print("EOS token ID:", EOS_TOKEN_ID)

# ============================================================
# 3. LOAD PROMPT DATA
# ============================================================

def load_jsonl(path):

    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    return records


train_records = load_jsonl(TRAIN_FILE)
val_records = load_jsonl(VAL_FILE)
test_records = load_jsonl(TEST_FILE)

print("\nLoaded:")
print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

# ============================================================
# 4. MASKED TOKENIZATION FUNCTION
# ============================================================

ASSISTANT_MARKER = "### ASSISTANT:\n"

def tokenize_example(example):

    prompt = example["prompt"]
    completion = example["completion"]

    if not prompt.endswith(ASSISTANT_MARKER):
        raise ValueError("Assistant marker missing or malformed.")

    # Tokenize prompt separately.
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True
    )["input_ids"]

    # Tokenize assistant response and EOS.
    response_ids = tokenizer(
        completion + EOS_TOKEN,
        add_special_tokens=False
    )["input_ids"]

    input_ids = prompt_ids + response_ids

    # Mask prompt tokens; train only on assistant response.
    labels = (
        [-100] * len(prompt_ids)
        + response_ids.copy()
    )

    # Do not silently truncate any feature values.
    if len(input_ids) > MAX_LENGTH:
        raise ValueError(
            f"Sequence length {len(input_ids)} exceeds "
            f"MAX_LENGTH={MAX_LENGTH}. No truncation permitted."
        )

    assert len(input_ids) == len(labels)
    assert all(x == -100 for x in labels[:len(prompt_ids)])
    assert labels[len(prompt_ids):] == response_ids
    assert labels[-1] == EOS_TOKEN_ID

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "length": len(input_ids),
        "prompt_length": len(prompt_ids),
        "response_length": len(response_ids),
        "label_id": int(example["label_id"])
    }


# ============================================================
# 5. PROCESS DATASETS
# ============================================================

def tokenize_dataset(records, split_name):

    tokenized = []

    for example in tqdm(records, desc=f"Tokenizing {split_name}"):

        tokenized.append(tokenize_example(example))

    return Dataset.from_list(tokenized)


train_dataset = tokenize_dataset(train_records, "Train")
val_dataset = tokenize_dataset(val_records, "Validation")
test_dataset = tokenize_dataset(test_records, "Test")

# ============================================================
# 6. VERIFY MASKING
# ============================================================

print("\n" + "=" * 65)
print("MASKING VERIFICATION")
print("=" * 65)

sample = train_dataset[0]

input_ids = sample["input_ids"]
labels = sample["labels"]

supervised_positions = [
    i for i, label in enumerate(labels)
    if label != -100
]

masked_positions = [
    i for i, label in enumerate(labels)
    if label == -100
]

print("Total tokens:", len(input_ids))
print("Masked prompt tokens:", len(masked_positions))
print("Supervised response tokens:", len(supervised_positions))

assert len(supervised_positions) > 0
assert len(masked_positions) > 0

# Decode the supervised portion
supervised_ids = [
    labels[i] for i in supervised_positions
]

print("\nDecoded supervised response:")
print(tokenizer.decode(supervised_ids))

# ============================================================
# 7. VERIFY ALL EXAMPLES
# ============================================================

def verify_dataset(dataset, split_name):

    total = len(dataset)
    oversized = 0
    missing_supervision = 0
    incorrect_eos = 0
    invalid_masks = 0

    lengths = []

    for example in tqdm(dataset, desc=f"Verifying {split_name}"):

        ids = example["input_ids"]
        labels = example["labels"]

        lengths.append(len(ids))

        if len(ids) > MAX_LENGTH:
            oversized += 1

        if not any(x != -100 for x in labels):
            missing_supervision += 1

        if labels[-1] != EOS_TOKEN_ID:
            incorrect_eos += 1

        if len(ids) != len(labels):
            invalid_masks += 1

    print(f"\n{split_name}:")
    print("Examples:", total)
    print("Oversized:", oversized)
    print("Missing supervision:", missing_supervision)
    print("Incorrect EOS:", incorrect_eos)
    print("Invalid label lengths:", invalid_masks)
    print("Maximum length:", max(lengths))
    print("Mean length:", round(np.mean(lengths), 2))

    assert oversized == 0
    assert missing_supervision == 0
    assert incorrect_eos == 0
    assert invalid_masks == 0

    return lengths


train_lengths = verify_dataset(train_dataset, "TRAIN")
val_lengths = verify_dataset(val_dataset, "VALIDATION")
test_lengths = verify_dataset(test_dataset, "TEST")

# ============================================================
# 8. SAVE TOKENIZED DATASETS
# ============================================================

train_dataset.save_to_disk(f"{OUTPUT_DIR}/train")
val_dataset.save_to_disk(f"{OUTPUT_DIR}/validation")
test_dataset.save_to_disk(f"{OUTPUT_DIR}/test")

print("\n" + "=" * 65)
print("STEP 3 COMPLETED")
print("=" * 65)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("Maximum sequence length:", MAX_LENGTH)
print("Prompt masking: Enabled")
print("Assistant-only loss: Enabled")
print("EOS supervision: Enabled")
print("Truncation: Disabled")

print("\nSaved to:", OUTPUT_DIR)

Loading tokenizer...
EOS token: '<｜end▁of▁sentence｜>'
EOS token ID: 32014

Loaded:
Train: 17693
Validation: 5898
Test: 5898


Tokenizing Train:   0%|          | 0/17693 [00:00<?, ?it/s]

Tokenizing Validation:   0%|          | 0/5898 [00:00<?, ?it/s]

Tokenizing Test:   0%|          | 0/5898 [00:00<?, ?it/s]


MASKING VERIFICATION
Total tokens: 2918
Masked prompt tokens: 2906
Supervised response tokens: 12

Decoded supervised response:
Predicted Malware Family: RedLineStealer<｜end▁of▁sentence｜>


Verifying TRAIN:   0%|          | 0/17693 [00:00<?, ?it/s]

... (5 outputs left out)

Saving the dataset (0/2 shards):   0%|          | 0/17693 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5898 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5898 [00:00<?, ? examples/s]


STEP 3 COMPLETED
Train: 17693
Validation: 5898
Test: 5898
Maximum sequence length: 4096
Prompt masking: Enabled
Assistant-only loss: Enabled
EOS supervision: Enabled
Truncation: Disabled

Saved to: /kaggle/working/deepseek_tokenized


In [1]:
import torch
import os

print("=" * 65)
print("GPU ENVIRONMENT CHECK")
print("=" * 65)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):

        props = torch.cuda.get_device_properties(i)

        print(f"\nGPU {i}: {props.name}")
        print("Total VRAM:", round(props.total_memory / 1024**3, 2), "GB")
        print("Compute capability:", props.major, props.minor)

        print(
            "Currently allocated:",
            round(torch.cuda.memory_allocated(i) / 1024**3, 2),
            "GB"
        )

        print(
            "Currently reserved:",
            round(torch.cuda.memory_reserved(i) / 1024**3, 2),
            "GB"
        )

    print("\nBF16 supported:", torch.cuda.is_bf16_supported())

else:
    print("WARNING: No CUDA GPU detected.")

print("\n" + "=" * 65)


GPU ENVIRONMENT CHECK
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU count: 2

GPU 0: Tesla T4
Total VRAM: 14.56 GB
Compute capability: 7 5
Currently allocated: 0.0 GB
Currently reserved: 0.0 GB

GPU 1: Tesla T4
Total VRAM: 14.56 GB
Compute capability: 7 5
Currently allocated: 0.0 GB
Currently reserved: 0.0 GB

BF16 supported: True



In [2]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [4]:
import torch
import transformers
import peft
import bitsandbytes as bnb
import accelerate
import datasets

print("=" * 65)
print("DEPENDENCY VERIFICATION")
print("=" * 65)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bnb.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2

print("\nAll required libraries imported successfully.")

DEPENDENCY VERIFICATION
PyTorch: 2.11.0+cu130
Transformers: 5.22.2
PEFT: 0.21.2
BitsAndBytes: 0.50.2
Accelerate: 1.15.0
Datasets: 5.2.0

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

All required libraries imported successfully.


In [5]:
import torch
import gc

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

# T4-compatible quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Quantization configuration created.")

# ============================================================
# 2. LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")

# ============================================================
# 3. LOAD MODEL
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.config.use_cache = False

print("\nModel loaded successfully.")

# ============================================================
# 4. PREPARE FOR QLORA
# ============================================================

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# ============================================================
# 5. CONFIGURE LORA
# ============================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

# ============================================================
# 6. TRAINABLE PARAMETER VERIFICATION
# ============================================================

print("\n" + "=" * 65)
print("LORA PARAMETER VERIFICATION")
print("=" * 65)

model.print_trainable_parameters()

# ============================================================
# 7. GPU MEMORY CHECK
# ============================================================

print("\n" + "=" * 65)
print("GPU MEMORY")
print("=" * 65)

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i) / 1024**3
    reserved = torch.cuda.memory_reserved(i) / 1024**3

    print(f"GPU {i}:")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved: {reserved:.2f} GB")

# ============================================================
# 8. FINAL CHECK
# ============================================================

assert model is not None
assert model.peft_config is not None

print("\nDeepSeek QLoRA initialization completed.")

Quantization configuration created.


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (4 outputs left out)

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]


Model loaded successfully.

LORA PARAMETER VERIFICATION
trainable params: 14,991,360 || all params: 1,361,463,296 || trainable%: 1.1011

GPU MEMORY
GPU 0:
Allocated: 0.52 GB
Reserved: 0.66 GB
GPU 1:
Allocated: 0.63 GB
Reserved: 0.78 GB

DeepSeek QLoRA initialization completed.
